# Hierarchical Data, the JSON Data Format, and APIs

In [1]:
import pandas as pd

## Shows Data

First we'll work with the "Girls" shows JSON data from the reading.

In [2]:
# Fetch data from a URL
import requests
response = requests.get("https://dlsun.github.io/pods/data/tvshows.json")

import json
data_shows = response.json()

In [3]:
df_shows = pd.json_normalize(data_shows)
df_shows

,id,url,name,type,language,genres,status,runtime,premiered,officialSite,...,externals.thetvdb,externals.imdb,image.medium,image.original,network,webChannel.id,webChannel.name,webChannel.country.name,webChannel.country.code,webChannel.country.timezone
0,139,http://www.tvmaze.com/shows/139/girls,Girls,Scripted,English,"[Drama, Romance]",Ended,30,2012-04-15,http://www.hbo.com/girls,...,220411,tt1723816,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
1,722,http://www.tvmaze.com/shows/722/the-golden-girls,The Golden Girls,Scripted,English,"[Drama, Comedy]",Ended,30,1985-09-14,None,...,71292,tt0088526,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
2,23542,http://www.tvmaze.com/shows/23542/good-girls,Good Girls,Scripted,English,"[Drama, Comedy, Crime]",Running,60,2018-02-26,https://www.nbc.com/good-girls?nbc=1,...,328577,tt6474378,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
3,6771,http://www.tvmaze.com/shows/6771/the-powerpuff...,The Powerpuff Girls,Animation,English,"[Comedy, Action, Science-Fiction]",Running,15,2016-04-04,https://www.cartoonnetwork.com/video/powerpuff...,...,307473,tt4718304,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
4,42726,http://www.tvmaze.com/shows/42726/florida-girls,Florida Girls,Scripted,English,[Comedy],Running,30,2019-07-10,https://poptv.com/floridagirls,...,363682,tt8548870,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
5,32087,http://www.tvmaze.com/shows/32087/chicken-girls,Chicken Girls,Scripted,English,"[Drama, Children, Music]",Running,16,2017-09-05,https://www.youtube.com/playlist?list=PLVewHiZ...,...,339854,None,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,274.0,Brat,United States,US,America/New_York
6,33320,http://www.tvmaze.com/shows/33320/derry-girls,Derry Girls,Scripted,English,[Comedy],Running,30,2018-01-04,http://www.channel4.com/programmes/derry-girls,...,338903,tt7120662,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
7,1955,http://www.tvmaze.com/shows/1955/the-powerpuff...,The Powerpuff Girls,Animation,English,"[Action, Children, Crime]",Ended,30,1998-11-18,None,...,76200,tt0175058,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
8,1073,http://www.tvmaze.com/shows/1073/bomb-girls,Bomb Girls,Scripted,English,"[Drama, Romance, War]",Ended,60,2012-01-04,None,...,254378,tt1955311,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
9,525,http://www.tvmaze.com/shows/525/gilmore-girls,Gilmore Girls,Scripted,English,"[Drama, Comedy, Romance]",Ended,60,2000-10-05,None,...,76568,tt0238784,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN


1\. Summarize the networks represented by these shows and the number of these shows that aired on each network.

In [4]:
#df_shows.groupby("network.name").count()

df_shows["network.name"].value_counts()

# print(type(data_shows), len(data_shows))

# print(df_shows.shape)




network.name
NBC                2
Cartoon Network    2
HBO                1
Pop                1
Channel 4          1
Global             1
The CW             1
Name: count, dtype: int64

2\. Find the number of seasons for each show in the data. Do this two ways: one which uses `df_shows`, and another that first flattens `data_shows` to a different data frame.

In [5]:
df_shows["number of seasons"] = df_shows["seasons"].apply(len)
df_shows[["name", "number of seasons"]]

,name,number of seasons
0,Girls,6
1,The Golden Girls,7
2,Good Girls,3
3,The Powerpuff Girls,3
4,Florida Girls,1
5,Chicken Girls,5
6,Derry Girls,2
7,The Powerpuff Girls,6
8,Bomb Girls,2
9,Gilmore Girls,8


In [23]:
df_seasons = pd.json_normalize(
    data_shows,
    record_path = "seasons", 
    meta = ["id", "name"], 
    meta_prefix = "show." 
    )


df_seasons.groupby(["show.id", "show.name"]).size().reset_index()

,show.id,show.name,0
0,139,Girls,6
1,525,Gilmore Girls,8
2,722,The Golden Girls,7
3,1073,Bomb Girls,2
4,1955,The Powerpuff Girls,6
5,6771,The Powerpuff Girls,3
6,23542,Good Girls,3
7,32087,Chicken Girls,5
8,33320,Derry Girls,2
9,42726,Florida Girls,1


3\. For each episode, find the length (number of characters) of the title. Then create summaries to answer: Which show tends to have the longest episode titles? The shortest?

In [7]:
df_episodes = pd.json_normalize(
    data_shows,
    record_path=["seasons", "episodes"],
    meta=["id", "name"],
    meta_prefix="show_"
)

df_episodes["title_length"] = df_episodes["name"].apply(len)

df_episodes[["name", "title_length"]]

,name,title_length
0,Pilot,5
1,Vagina Panic,12
2,All Adventurous Women Do,24
3,Hannah's Diary,14
4,Hard Being Easy,15
...,...,...
736,Hay Bale Maze,13
737,It's Just Like Riding a Bike,28
738,Lorelai? Lorelai?,17
739,Unto the Breach,15


4\. Do any cast members in the data set share a birthday with you? Who, and what show are they on? (If there isn't anyone, try a different day.)

In [8]:
df_birthdays = pd.json_normalize(
    data_shows,
    record_path=["cast"],
    meta=["id", "name",],
    meta_prefix="show_"
)

df_birthdays = df_birthdays[
    ["person.name", "person.birthday", "show_name"]
]

df_birthdays[(df_birthdays["person.birthday"] == "1983-11-19")]


# type(data_shows)
# type(data_shows[0])
# data_shows[0].keys()
# #data_shows[0]["cast"][0]["person"]
# data_shows[0]["cast"][0]["person"]

,person.name,person.birthday,show_name
4,Adam Driver,1983-11-19,Girls


I had to pick a random birthday because no one had mine but I did 1983-11-19, which is Adam Driver'birthday and he was on the show Girls. 

## TVMaze API

Now you will work with the [TVMaze API](http://www.tvmaze.com/api) from the reading. Use the API to request JSON data that you can use to answer the following questions.

1\. What was the longest show that aired in the U.S. on February 4, 2018?

_Hint:_ Use the ["Schedule" endpoint](http://www.tvmaze.com/api#schedule) to first get the data for all shows that aired on that date.

In [9]:
import requests

response = requests.get(
    "https://api.tvmaze.com/schedule",
    params={"country": "US", "date": "2018-02-04"},
)

print(response.status_code)

200


In [10]:
schedule = response.json()

In [11]:
df_schedule = pd.json_normalize(schedule)

In [12]:
df_schedule.sort_values(
    "runtime",
    ascending=False
)[["show.name", "runtime", "airtime"]].head(2)

,show.name,runtime,airtime
23,Super Bowl,210,18:30
29,Alaska: The Last Frontier,180,20:00


2\. Among all shows that aired in the U.S. on Feburary 4, 2018, which non-voice actors appeared on more than one show? Note: some people are credited with multiple rows on the same show and thus appear as multiple rows in the data frame.

Hint: You will need to write a for loop to make multiple requests. Use the "show.id" from the data from part 1, and use the ["Shows" endpoint](http://www.tvmaze.com/api#show-cast) to get the cast of each show. Don't forget to stagger your requests, or you will be blocked by the website!

In [13]:
show_ids = df_schedule["show.id"].unique()
len(show_ids)


38

In [14]:
# https://api.tvmaze.com/shows/1/cast

import time

cast_records = []

for show_id in show_ids:
    response = requests.get(
        f"https://api.tvmaze.com/shows/{show_id}/cast"
    )

    if response.status_code != 200:
        print("Skipped", show_id, "Status error", response.status_code)
        continue

    members = response.json()

    for member in members:
        member["show_id"] = int(show_id)

    cast_records.extend(members)
    time.sleep(0.5)

In [24]:
df_cast = pd.json_normalize(cast_records)

show_lookup = df_schedule[
    ["show.id", "show.name"]
].drop_duplicates()

df_cast = df_cast.merge(
    show_lookup,
    left_on="show_id",
    right_on="show.id",
    how="left"
)

unique_non_voice = (
    df_cast[df_cast["voice"] == False]
    .drop_duplicates(subset=["person.id", "show_id"])
)

show_counts = (
    unique_non_voice
    .groupby(["person.id", "person.name"])["show_id"]
    .nunique()
    .reset_index(name="number_of_shows")
)

multi_show_actors = show_counts[
    show_counts["number_of_shows"] > 1
]

answer = (
    multi_show_actors
    .merge(
        unique_non_voice[
            ["person.id", "person.name", "show.name"]
        ],
        on=["person.id", "person.name"]
    )
    [["person.name", "show.name"]]
    .drop_duplicates()
    .sort_values(["person.name", "show.name"])
)

answer

,person.name,show.name
0,John Dickerson,CBS News Sunday Morning
1,John Dickerson,Face the Nation
2,Tom Llamas,ABC World News Tonight with David Muir
3,Tom Llamas,NBC Nightly News


# Tasty API

[Tasty.co](http://tasty.co) is a website and app that offers food recipes. They have made these recipes available through a [a REST API](https://rapidapi.com/apidojo/api/tasty). However, unlike the TVMaze API, this one requires authentication.

Specifically, you will need to create an account and subscribe to the "Basic" (free) plant. You will then be provided with an API key (X-Rapid-API-Key) that will need to be supplied with every request you make. This is used to track and limit usage.

1. [create an account](https://rapidapi.com/apidojo/api/tasty) and subscribe to the "Basic" (free) plan
2. log in and copy the X-RapidAPI-Key, which is a long string of letters and digits
3. paste this key to replace "PUT-YOUR-KEY-HERE" in the `headers` below

If you did everything correctly, then running the cell below should return a JSON object containing all the tags recognized by the Tasty API.

In [26]:
import requests

domain = "https://tasty.p.rapidapi.com"
endpoint = "tags/list"
url = f"{domain}/{endpoint}"

# TODO: Update the `headers` with your X-RapidAPI-Key.
headers = {
	"X-RapidAPI-Key": "98a1a38689msh828b02a57ecf516p106f60jsncd26bb67b1fa",
	"X-RapidAPI-Host": "tasty.p.rapidapi.com"
}

# Make an HTTP request to the REST API, get the JSON response.
response = requests.get(url, headers=headers)
response.json()

{'count': 588,
 'results': [{'root_tag_type': 'cuisine',
   'name': 'brazilian',
   'id': 64446,
   'parent_tag_name': 'central_south_american',
   'display_name': 'Brazilian',
   'type': 'central_south_american'},
  {'root_tag_type': 'cuisine',
   'name': 'german',
   'id': 64450,
   'parent_tag_name': 'european',
   'display_name': 'German',
   'type': 'european'},
  {'root_tag_type': 'healthy',
   'name': 'healthy',
   'id': 64466,
   'parent_tag_name': None,
   'display_name': 'Healthy',
   'type': 'healthy'},
  {'root_tag_type': 'dietary',
   'name': 'vegetarian',
   'id': 64469,
   'parent_tag_name': 'dietary',
   'display_name': 'Vegetarian',
   'type': 'dietary'},
  {'root_tag_type': 'seasonal',
   'name': 'christmas',
   'id': 64473,
   'parent_tag_name': 'holidays',
   'display_name': 'Christmas',
   'type': 'holidays'},
  {'root_tag_type': 'seasonal',
   'name': 'halloween',
   'id': 64476,
   'parent_tag_name': 'holidays',
   'display_name': 'Halloween',
   'type': 'holiday

You will need to pass these `headers` with every HTTP request to the API. The API key (X-RapidAPI-Key) is how the server keeps track of how many requests you have made.

Take a look at [the documentation](https://rapidapi.com/apidojo/api/tasty). We will use the recipes/list endpoint.

Make sure you are logged into the account you are created, and select the recipes/list endpoint from the menu at left. Notice that this brings up a form that you can fill in, which generates the corresponding code.  By default, it provides Node.js code; change this to Python with the Requests client.

1\. Search for recipes containing "daikon" (an Asian radish) and request the JSON data, and convert it to a Pandas data frame.

In [27]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
endpoint = "recipes/list"
url = f"{domain}/{endpoint}"

params = {
    "from": 0,
    "size": 40,
    "q": "daikon"
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

response.raise_for_status()

daikon_data = response.json()
df_daikon = pd.json_normalize(daikon_data["results"])

df_daikon

,nutrition_visibility,country,instructions,keywords,facebook_posts,language,seo_path,id,brand,slug,...,nutrition.calories,nutrition.sugar,show.name,show.id,total_time_tier.tier,total_time_tier.display_tier,brand.image_url,brand.name,brand.id,brand.slug
0,auto,US,"[{'start_time': 7666, 'appliance': None, 'end_...","beef bulgogi, creamy and crunchy foods, easy b...",[],eng,"8757513,9295873,64455",6046,NaN,instant-pot-beef-bulgogi,...,1137.0,40.0,Tasty,17,NaN,NaN,NaN,NaN,NaN,NaN
1,auto,US,"[{'start_time': 9666, 'appliance': None, 'end_...",None,[],eng,"8757513,9295873,64461",4490,NaN,top-chef-junior-pork-banh-mi-burger,...,905.0,34.0,Tasty,17,under_30_minutes,Under 30 minutes,NaN,NaN,NaN,NaN
2,auto,US,"[{'start_time': 1000, 'appliance': None, 'end_...",,[],eng,"8757513,9295873,64461",5643,NaN,banh-mi-meatball-sandwich,...,462.0,17.0,Tasty,17,under_2_hours,Under 2 hours,NaN,NaN,NaN,NaN
3,auto,US,"[{'start_time': 0, 'appliance': None, 'end_tim...","chicken banh mi, do chua recipe, easy weeknigh...",[],eng,"8757513,9295873,64461",9820,NaN,grilled-chicken-banh-mi,...,1026.0,22.0,Tasty,17,NaN,NaN,NaN,NaN,NaN,NaN
4,auto,US,"[{'start_time': 5833, 'appliance': None, 'hack...","tasty, tasty_vegetarian",[],eng,"8757513,9295873,64455",4258,NaN,how-to-make-kimchi,...,170.0,9.0,Tasty: Tasty Vegetarian,49,under_2.5_hours,Under 2.5 hours,NaN,NaN,NaN,NaN
5,auto,US,"[{'start_time': 423000, 'appliance': None, 'ha...",,[],eng,"8757513,9295873,64454",7224,NaN,japanese-omelette,...,412.0,8.0,Tasty 101,63,under_15_minutes,Under 15 minutes,NaN,NaN,NaN,NaN
6,auto,US,"[{'start_time': 8000, 'appliance': None, 'end_...","chicken, dairy-free, dinner, gluten-free, heal...",[],eng,"8757513,9295873,64460",4621,NaN,low-carb-pad-thai,...,310.0,10.0,Goodful,34,under_45_minutes,Under 45 minutes,NaN,NaN,NaN,NaN
7,auto,US,"[{'start_time': 0, 'appliance': None, 'hacks':...","breakfast, chinese, hawker food, international...",[],eng,"9295813,64486,64459,9299649",6542,NaN,fried-carrot-cake,...,196.0,2.0,Tasty,17,NaN,NaN,https://img.buzzfeed.com/tasty-app-user-assets...,VisitSingapore,24.0,visitsingapore
8,auto,US,"[{'start_time': 7166, 'appliance': None, 'end_...","beef, beef bulgogi, bibimbap, bulgogi, buzzfee...",[],eng,"8757513,9295873,64455",4013,NaN,bibimbap-by-chef-esther-choi,...,1382.0,13.0,Tasty,17,under_1.5_hours,Under 1.5 hours,NaN,NaN,NaN,NaN
9,auto,US,"[{'start_time': 60000, 'appliance': None, 'end...","banh mi recipe, daikon, grilled lemongrass por...",[],eng,"8757513,9295873,64461",5152,NaN,grilled-lemongrass-pork-banh-mi,...,1224.0,26.0,Tasty,17,NaN,NaN,NaN,NaN,NaN,NaN


2\. How many recipes containing daikon are there? Which one is the cheapest per portion?

In [28]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
print("Number of daikon recipes:", daikon_data["count"])

Number of daikon recipes: 16


In [29]:
df_daikon["price.portion"] = pd.to_numeric(
    df_daikon["price.portion"],
    errors="coerce"
)

cheapest_daikon = (
    df_daikon
    .dropna(subset=["price.portion"])
    .sort_values("price.portion")
    [["name", "price.portion"]]
    .head(1)
)

cheapest_daikon

,name,price.portion
10,Vegan Tofu Bao Buns With Pickled Vegetables,300.0


3\. Find recipes containing avocado and request the JSON data.

**Hint:** Note that there are hundreds of results, but the API only returns 20 results by default and only 40 results maximum, even if you specify the `size=` parameter, so you will need to use a `for` loop, incrementing the `from=` parameter. Be sure to respect the API's rate limits (or you may be blocked!)

**Note:** Recall from the example in the reading that we created an empty list `episodes = []` and then added the results of each request to it in the for loop with `episodes.extend(response.json())`. But note that in the recipes/list endpoint there are two keys: count and results. We only want the results so try instead `.extend(response.json()).get("results")`.

**Suggestion:** Try writing a loop that only makes 2 or 3 requests first so you can test that it's working correctly. Also, make sure you use separate cells for code that makes requests to the API versus processing of the results; you don't want to rerun the requests unless absolutely necessary!

In [37]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
import time

avocado_recipes = []

params = {
    "from": 0,
    "size": 40,
    "q": "avocado"
}

response = requests.get(
    url,
    headers=headers,
    params=params
)

response.raise_for_status()

first_page = response.json()
total_avocado_results = first_page["count"]

avocado_recipes.extend(first_page["results"])

for start in range(40, total_avocado_results, 40):
    params = {
        "from": start,
        "size": 40,
        "q": "avocado"
    }

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    response.raise_for_status()

    avocado_recipes.extend(
        response.json().get("results", [])
    )

    time.sleep(1)

print("Total reported by API:", total_avocado_results)
print("Recipes downloaded:", len(avocado_recipes))

Total reported by API: 444
Recipes downloaded: 444


In [44]:
df_avocado = pd.json_normalize(avocado_recipes)
df_avocado.head()

,nutrition_visibility,country,instructions,keywords,facebook_posts,language,seo_path,id,brand,slug,...,show.name,show.id,total_time_tier.tier,total_time_tier.display_tier,total_time_tier,brand.image_url,brand.name,brand.id,brand.slug,recipes
0,auto,US,"[{'start_time': 0, 'appliance': None, 'end_tim...",", avocado, buzzfeedtasty, carbonara, dinner, e...",[],eng,"8757513,9295874,64453",56,NaN,avocado-carbonara,...,Tasty,17,under_30_minutes,Under 30 minutes,NaN,NaN,NaN,NaN,NaN,NaN
1,auto,ZZ,"[{'start_time': 0, 'appliance': 'oven', 'end_t...",None,[],eng,"9295813,64486,64459",1340,NaN,avocado-lime-salmon,...,Tasty,17,under_30_minutes,Under 30 minutes,NaN,NaN,NaN,NaN,NaN,NaN
2,auto,US,"[{'start_time': 4833, 'appliance': None, 'hack...","avocado, cucumber, easy, quick, quinoa, salad,...",[],eng,"9295813,64489,9299495",3932,NaN,avocado-quinoa-power-salad,...,Tasty: Tasty Vegetarian,49,under_1_hour,Under 1 hour,NaN,NaN,NaN,NaN,NaN,NaN
3,auto,US,"[{'start_time': 0, 'appliance': 'oven', 'end_t...","avocado, ceviche, cilantro, citrus, cocktail, ...",[],eng,"8757513,64444,64457",2819,NaN,shrimp-avocado-tostadas,...,Goodful,34,under_45_minutes,Under 45 minutes,NaN,NaN,NaN,NaN,NaN,NaN
4,auto,US,"[{'start_time': 0, 'appliance': None, 'hacks':...","tasty, tasty_vegetarian",[],eng,"9295813,64489,9299495",232,NaN,cucumber-tomato-and-avocado-salad,...,Tasty,17,under_15_minutes,Under 15 minutes,NaN,NaN,NaN,NaN,NaN,NaN


4\. For the recipes containing avocado, compute the proportion of reviews that are positive. For the avocado recipes with over 500 reviews, which one has the highest proportion of positive reviews?


In [46]:
df_avocado["positive_reviews"] = pd.to_numeric(
    df_avocado["user_ratings.count_positive"],
    errors="coerce"
).fillna(0)

df_avocado["negative_reviews"] = pd.to_numeric(
    df_avocado["user_ratings.count_negative"],
    errors="coerce"
).fillna(0)

df_avocado["total_reviews"] = (
    df_avocado["positive_reviews"]
    + df_avocado["negative_reviews"]
)

df_avocado["positive_proportion"] = (
    df_avocado["positive_reviews"]
    / df_avocado["total_reviews"]
)

answer = (
    df_avocado[df_avocado["total_reviews"] > 500]
    .sort_values("positive_proportion", ascending=False)
    [[
        "name",
        "positive_reviews",
        "negative_reviews",
        "total_reviews",
        "positive_proportion"
    ]]
    .head(1)
)

answer

,name,positive_reviews,negative_reviews,total_reviews,positive_proportion
13,Grilled Salmon With Avocado Salsa,1874.0,24.0,1898.0,0.987355


5\. Take the avocado JSON data from above (you do NOT need to read in the data from the REST API again). How many recipes are vegetarian? You should be able to identify this from the "tags" attribute.

Hint: Try using `json_normalize` with "tags" as the record path to flatten the data so that there is one row for each tag.

In [47]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
df_avocado_tags = pd.json_normalize(
    avocado_recipes,
    record_path="tags",
    meta=["id", "name"],
    meta_prefix="recipe_"
)

vegetarian_count = (
    df_avocado_tags[
        df_avocado_tags["name"] == "vegetarian"
    ]["recipe_id"]
    .nunique()
)

print(
    "Number of vegetarian avocado recipes:",
    vegetarian_count
)

Number of vegetarian avocado recipes: 207
